# ChromoDiff-Absorb: Generative Zero-Shot Pathogenicity Prediction via Discrete Genomic Diffusion

This notebook implements **ChromoDiff** (formalized as ChromoDiff-Absorb), a categorical 1D dilated residual diffusion model that learns the non-coding syntax of the human genome. It contains:
1. **Automatic Project Builder**: Writes out all the modular Python files (`src/` and `configs/`) so that the project is completely self-contained and modular on your Kaggle instance.
2. **Data Preprocessor**: Downloads `clinvar.vcf` and `hg38.fa`, filters SNPs, and tokenizes sequence windows (or generates high-quality synthetic data for fast dry-runs).
3. **Model Training**: Denoises discrete sequences by learning to reconstruct masked bases with a selective cross-entropy loss.
4. **Zero-Shot GVES Scoring**: Computes the Generative Variant Effect Score (GVES) at evaluation time to predict ClinVar pathogenicity.

## Setup & GPU Check

In [ ]:
# 1. Verify GPU acceleration availability on Kaggle
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name  :", torch.cuda.get_device_name(0))

# 2. Install required package dependency for Fasta reading
!pip install -q pyfaidx pandas numpy tqdm pyyaml matplotlib scikit-learn seaborn datasets

## Step 1: Write Modular Project Code

Run the cell below to write out the package directories and files.

In [ ]:
import os

# Create directory structures
os.makedirs("configs", exist_ok=True)
os.makedirs("src/models", exist_ok=True)

# Write configs/base_config.yaml
with open("configs/base_config.yaml", "w", encoding="utf-8") as f:
    f.write("""# Diffusion Schedule Settings
T_STEPS: 1000
BETA_START: 0.0001
BETA_END: 0.02
VOCAB_SIZE: 6                # A=0, C=1, G=2, T=3, N=4, [MASK]=5
SEQ_LEN: 1024
MIN_CORRUPTION_RATE: 0.0     # Floor for token masking during diffusion

# Model Architecture
HIDDEN_DIM: 256

# Training Hyperparameters
BATCH_SIZE: 64
EPOCHS: 50
LEARNING_RATE: 0.001         # Base learning rate
GRAD_CLIP: 1.0
WARMUP_EPOCHS: 2             # Linear warmup duration (was 5 — too long)
T_0: 25                      # CosineAnnealingWarmRestarts restart cycle
ETA_MIN: 0.000001            # Floor for Cosine learning rate

# Paths & Directories
DATA_DIR: "data/processed"
CHECKPOINT_DIR: "outputs/checkpoints"
SEED: 42
""")

# Write src/utils.py
with open("src/utils.py", "w", encoding="utf-8") as f:
    f.write("""import os
import yaml
import torch
import random
import logging
import numpy as np

def set_seed(seed: int = 42):
    \"\"\"
    Set seeds for random, numpy, and torch for maximum reproducibility.
    \"\"\"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def load_config(config_path: str) -> dict:
    \"\"\"
    Load YAML configuration file.
    \"\"\"
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Configuration file not found: {config_path}")
    with open(config_path, "r") as f:
        return yaml.safe_load(f)

def setup_logger(name: str = "ChromoDiff") -> logging.Logger:
    \"\"\"
    Configure a standard console logger.
    \"\"\"
    logger = logging.getLogger(name)
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        ch = logging.StreamHandler()
        ch.setLevel(logging.INFO)
        formatter = logging.Formatter(
            "[%(asctime)s][%(name)s][%(levelname)s] %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S"
        )
        ch.setFormatter(formatter)
        logger.addHandler(ch)
    return logger

def setup_dirs(*dirs):
    \"\"\"
    Ensure directories exist.
    \"\"\"
    for d in dirs:
        if d:
            os.makedirs(d, exist_ok=True)
""")

# Write src/dataset.py
with open("src/dataset.py", "w", encoding="utf-8") as f:
    f.write("""import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class GenomicDataset(Dataset):
    \"\"\"
    Wraps a tensor of genomic sequences represented as token indices.
    \"\"\"
    def __init__(self, data: torch.Tensor):
        if not isinstance(data, torch.Tensor):
            self.data = torch.tensor(data, dtype=torch.long)
        else:
            self.data = data.long()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def reverse_complement_tokens(x: np.ndarray) -> np.ndarray:
    \"\"\"
    Reverse complement sequence array in NumPy.
    Mapping: A(0)->T(3), C(1)->G(2), G(2)->C(1), T(3)->A(0), N(4)->N(4), [MASK](5)->[MASK](5)
    \"\"\"
    comp_map = np.array([3, 2, 1, 0, 4, 5], dtype=np.int8)
    return comp_map[x[:, ::-1]]

def reverse_complement_tensor(x: torch.Tensor) -> torch.Tensor:
    \"\"\"
    Reverse complement sequence tensor in PyTorch.
    \"\"\"
    comp_map = torch.tensor([3, 2, 1, 0, 4, 5], dtype=torch.long, device=x.device)
    reversed_x = torch.flip(x, dims=[-1])
    return comp_map[reversed_x]

def get_dataloader(data_path: str, batch_size: int, shuffle: bool = True, num_workers: int = 0) -> DataLoader:
    \"\"\"
    Load sequence token array from disk, wrap it in a GenomicDataset, and return a DataLoader.
    \"\"\"
    data_np = np.load(data_path)
    data_tensor = torch.tensor(data_np, dtype=torch.long)
    dataset = GenomicDataset(data_tensor)
    
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=True,
        pin_memory=torch.cuda.is_available(),
        num_workers=num_workers
    )
    return loader
""")

# Write src/diffusion.py
with open("src/diffusion.py", "w", encoding="utf-8") as f:
    f.write("""import torch

class AbsorbingStateScheduler:
    \"\"\"
    Scheduler for absorbing-state discrete genomic diffusion (D3PM-Absorb).
    Defines beta/alpha cumulative product schedules and injects noise by replacing tokens with [MASK] (5).
    \"\"\"
    def __init__(self, num_steps: int = 1000, beta_start: float = 1e-4, beta_end: float = 0.02, min_corruption_rate: float = 0.15):
        self.num_steps = num_steps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.min_corruption_rate = min_corruption_rate

        # Initialize linear beta schedule
        self.betas = torch.linspace(beta_start, beta_end, num_steps)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)

    def to(self, device: torch.device):
        self.betas = self.betas.to(device)
        self.alphas = self.alphas.to(device)
        self.alphas_cumprod = self.alphas_cumprod.to(device)
        return self

    def sample_timesteps(self, batch_size: int, device: torch.device) -> torch.Tensor:
        \"\"\"
        Importance-weighted timestep sampling.
        Uses a squared distribution to bias toward low/mid corruption levels
        where the model can actually learn sequence context from surrounding bases.
        High t (>800) masks >90% of tokens, leaving no context to learn from.
        \"\"\"
        u = torch.rand(batch_size, device=device)
        t = (u ** 2 * self.num_steps).long().clamp(0, self.num_steps - 1)
        return t

    def q_sample(self, x_start: torch.Tensor, t: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        \"\"\"
        Sample noisy sequence x_t from clean x_0.
        
        Each base either:
          - survives unchanged   with prob ᾱ_t  (but never more than 1 - min_corruption_rate)
          - is replaced by [MASK] (5) with prob max(1 - ᾱ_t, min_corruption_rate)
        \"\"\"
        B, L = x_start.shape
        device = x_start.device
        
        if self.alphas_cumprod.device != device:
            self.alphas_cumprod = self.alphas_cumprod.to(device)

        # Get alphas_cumprod for the given step
        a_bar = self.alphas_cumprod[t].unsqueeze(1) # [B, 1]

        # Clamp survival probability so corruption rate (masking) >= min_corruption_rate (e.g. 15%)
        a_bar_floored = torch.clamp(a_bar, max=1.0 - self.min_corruption_rate)

        rand_probs = torch.rand((B, L), device=device)
        mutate_mask = rand_probs > a_bar_floored # [B, L] Bool

        # Replace selected bases with [MASK] (5)
        x_noisy = torch.where(mutate_mask, torch.tensor(5, device=device, dtype=torch.long), x_start)

        return x_noisy, mutate_mask
""")

# Write src/models/embedding.py
with open("src/models/embedding.py", "w", encoding="utf-8") as f:
    f.write("""import math
import torch
import torch.nn as nn

class SinusoidalPositionEmbeddings(nn.Module):
    \"\"\"
    Encodes discrete scalar timestep t -> [Batch, dim] sinusoidal dense vector.
    Allows the model blocks to adapt reconstruction depending on corruption rate.
    \"\"\"
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, time: torch.Tensor) -> torch.Tensor:
        device = time.device
        half_dim = self.dim // 2
        freq = math.log(10000) / (half_dim - 1)
        freq = torch.exp(torch.arange(half_dim, device=device) * -freq)
        angles = time[:, None].float() * freq[None, :]
        return torch.cat([angles.sin(), angles.cos()], dim=-1)
""")

# Write src/models/unet.py
with open("src/models/unet.py", "w", encoding="utf-8") as f:
    f.write("""import math
import torch
import torch.nn as nn
from .embedding import SinusoidalPositionEmbeddings

class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_len: int = 2048):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)
        
        t = torch.arange(max_len, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer('cos_cached', emb.cos(), persistent=False)
        self.register_buffer('sin_cached', emb.sin(), persistent=False)
        
    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        seq_len = x.shape[1]
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    cos = cos.unsqueeze(0).unsqueeze(2)
    sin = sin.unsqueeze(0).unsqueeze(2)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

class RoPESelfAttention(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.qkv_proj = nn.Linear(hidden_dim, hidden_dim * 3)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.rope = RotaryEmbedding(self.head_dim)
        self.norm = nn.GroupNorm(8, hidden_dim)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, L = x.shape
        h = x.permute(0, 2, 1)
        
        qkv = self.qkv_proj(h)
        qkv = qkv.reshape(B, L, 3, self.num_heads, self.head_dim)
        q, k, v = qkv[:, :, 0], qkv[:, :, 1], qkv[:, :, 2]
        
        cos, sin = self.rope(q)
        cos = cos.to(q.device)
        sin = sin.to(q.device)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        
        q = q.permute(0, 2, 1, 3)
        k = k.permute(0, 2, 1, 3)
        v = v.permute(0, 2, 1, 3)
        
        attn_weights = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.head_dim)
        attn_probs = torch.softmax(attn_weights, dim=-1)
        
        attn_out = torch.matmul(attn_probs, v)
        attn_out = attn_out.permute(0, 2, 1, 3).reshape(B, L, C)
        
        out = h + self.out_proj(attn_out)
        out = out.permute(0, 2, 1)
        return self.norm(out)

class DilatedResidualBlock(nn.Module):
    def __init__(self, hidden_dim: int, dilation: int):
        super().__init__()
        self.conv1 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=dilation, dilation=dilation)
        self.norm1 = nn.GroupNorm(8, hidden_dim)
        self.act1 = nn.GELU()
        
        self.time_proj = nn.Linear(hidden_dim, hidden_dim)
        
        self.conv2 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=dilation, dilation=dilation)
        self.norm2 = nn.GroupNorm(8, hidden_dim)
        
    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.conv1(x)
        h = self.act1(self.norm1(h))
        t_proj = self.time_proj(t_emb).unsqueeze(2)
        h = h + t_proj
        h = self.act1(self.norm2(self.conv2(h)))
        return x + h

class GenoDiff1D(nn.Module):
    def __init__(self, vocab_size: int = 6, hidden_dim: int = 256, dilations: list = None):
        super().__init__()
        if dilations is None:
            dilations = [1, 2, 4, 8, 16, 32]

        self.dna_embedding = nn.Embedding(vocab_size, hidden_dim)

        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim),
        )

        self.res_blocks1 = nn.ModuleList([
            DilatedResidualBlock(hidden_dim, dilation=d) for d in [1, 2, 4, 8]
        ])
        self.attn = RoPESelfAttention(hidden_dim, num_heads=4)
        self.res_blocks2 = nn.ModuleList([
            DilatedResidualBlock(hidden_dim, dilation=d) for d in [16, 32]
        ])

        self.output_norm = nn.GroupNorm(8, hidden_dim)
        self.final_conv = nn.Conv1d(hidden_dim, vocab_size - 1, kernel_size=1)

    def forward(self, noisy_dna: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        x = self.dna_embedding(noisy_dna).permute(0, 2, 1)
        t_emb = self.time_mlp(t)

        for block in self.res_blocks1:
            x = block(x, t_emb)
            
        x = self.attn(x)

        for block in self.res_blocks2:
            x = block(x, t_emb)

        logits = self.final_conv(self.output_norm(x))
        return logits
""")

# Write src/models/__init__.py
with open("src/models/__init__.py", "w", encoding="utf-8") as f:
    f.write("""from .embedding import SinusoidalPositionEmbeddings
from .unet import DilatedResidualBlock, GenoDiff1D
""")

# Write src/train.py
with open("src/train.py", "w", encoding="utf-8") as f:
    f.write("""import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm

from .utils import set_seed, load_config, setup_logger, setup_dirs
from .dataset import get_dataloader, reverse_complement_tensor
from .diffusion import AbsorbingStateScheduler
from .models.unet import GenoDiff1D

def get_lr(optimizer):
    return optimizer.param_groups[0]["lr"]

def linear_warmup(step, warmup_steps, base_lr):
    \"\"\"Scale LR linearly from 0 -> base_lr over warmup_steps.\"\"\"
    return base_lr * min(1.0, step / max(warmup_steps, 1))

def reverse_complement_logits(logits: torch.Tensor) -> torch.Tensor:
    comp_map = [3, 2, 1, 0, 4]
    reversed_logits = torch.flip(logits, dims=[-1])
    return reversed_logits[:, comp_map, :]

def train_model(config_path: str):
    # 1. Load config and setup utils
    config = load_config(config_path)
    set_seed(config.get("SEED", 42))
    logger = setup_logger("ChromoDiff.Train")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    
    checkpoint_dir = config.get("CHECKPOINT_DIR", "outputs/checkpoints")
    data_dir = config.get("DATA_DIR", "data/processed")
    setup_dirs(checkpoint_dir)
    
    # 2. Get data loader
    train_data_path = os.path.join(data_dir, "X_healthy.npy")
    if not os.path.exists(train_data_path):
        logger.error(f"Training data not found at {train_data_path}. Please run preprocessing first.")
        return
        
    logger.info(f"Loading data from {train_data_path}...")
    train_loader = get_dataloader(
        data_path=train_data_path,
        batch_size=config["BATCH_SIZE"],
        shuffle=True,
        num_workers=0
    )
    
    # 3. Initialize model and scheduler
    vocab_size = config.get("VOCAB_SIZE", 6)
    hidden_dim = config.get("HIDDEN_DIM", 256)
    model = GenoDiff1D(vocab_size=vocab_size, hidden_dim=hidden_dim).to(device)
    
    num_steps = config.get("T_STEPS", 1000)
    scheduler_diffusion = AbsorbingStateScheduler(
        num_steps=num_steps,
        beta_start=config.get("BETA_START", 1e-4),
        beta_end=config.get("BETA_END", 0.02),
        min_corruption_rate=config.get("MIN_CORRUPTION_RATE", 0.15)
    ).to(device)
    
    # 4. Optimizer and LR Scheduler setup
    optimizer = optim.AdamW(model.parameters(), lr=config["LEARNING_RATE"], weight_decay=1e-4)
    
    lr_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=config.get("T_0", 25),
        T_mult=1,
        eta_min=config.get("ETA_MIN", 1e-6)
    )
    
    # Mixed precision setup
    use_amp = torch.cuda.is_available()
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    
    best_loss = float("inf")
    train_history = []
    
    warmup_epochs = config.get("WARMUP_EPOCHS", 2)
    warmup_steps = warmup_epochs * len(train_loader)
    global_step = 0
    epochs = config.get("EPOCHS", 50)
    
    logger.info("Starting Unsupervised Diffusion Training...")
    logger.info(f"  Model params : {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")
    logger.info(f"  LR warmup    : {warmup_epochs} epochs ({warmup_steps} steps)")
    logger.info(f"  LR restarts  : every {config.get('T_0', 25)} epochs")
    logger.info(f"  RC augment   : 50% per batch")
    logger.info(f"  Timestep samp: importance-weighted (squared)")
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{epochs:02d}")
        for batch_idx, x_start in enumerate(progress_bar):
            # Apply linear learning rate warmup
            if global_step < warmup_steps:
                warm_lr = linear_warmup(global_step, warmup_steps, config["LEARNING_RATE"])
                for pg in optimizer.param_groups:
                    pg["lr"] = warm_lr
                    
            x_start = x_start.to(device, non_blocking=True)
            
            # Reverse complement augmentation (50% chance per batch)
            if torch.rand(1).item() < 0.5:
                x_start = reverse_complement_tensor(x_start)
            
            # Sample importance-weighted diffusion timesteps (biased toward low/mid t)
            t = scheduler_diffusion.sample_timesteps(x_start.shape[0], device)
            
            # Apply forward diffusion (adds [MASK] tokens)
            x_noisy, mutate_mask = scheduler_diffusion.q_sample(x_start, t)
            
            optimizer.zero_grad()
            
            # Run model with autocast
            with torch.amp.autocast("cuda", enabled=use_amp):
                predicted_logits = model(x_noisy, t)
                
                # Masked cross entropy loss (vocabulary size is vocab_size - 1 = 5)
                mask_flat = mutate_mask.view(-1)
                logits_flat = predicted_logits.permute(0, 2, 1).reshape(-1, vocab_size - 1)
                labels_flat = x_start.view(-1)
                
                logits_masked = logits_flat[mask_flat]
                labels_masked = labels_flat[mask_flat]
                
                if mask_flat.sum() > 0:
                    loss_ce = F.cross_entropy(logits_masked, labels_masked)
                else:
                    loss_ce = F.cross_entropy(logits_flat, labels_flat)
                
                # Double-strand consistency loss
                x_noisy_rc = reverse_complement_tensor(x_noisy)
                predicted_logits_rc = model(x_noisy_rc, t)
                target_logits_rc = reverse_complement_logits(predicted_logits)
                loss_dsc = F.mse_loss(predicted_logits_rc, target_logits_rc)
                
                loss = loss_ce + 0.1 * loss_dsc
            
            # Backpropagation using gradient scaling
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.get("GRAD_CLIP", 1.0))
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += loss.item()
            global_step += 1
            
            progress_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "lr": f"{get_lr(optimizer):.2e}"
            })
            
        # Step the learning rate scheduler (after warmup phase)
        if global_step >= warmup_steps:
            lr_scheduler.step(epoch - warmup_epochs + 1)
            
        avg_loss = epoch_loss / len(train_loader)
        train_history.append(avg_loss)
        
        logger.info(f"Epoch {epoch:02d} | Avg Loss: {avg_loss:.4f} | LR: {get_lr(optimizer):.2e}")
        
        # Save epoch checkpoint
        ckpt = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "avg_loss": avg_loss,
        }
        torch.save(ckpt, os.path.join(checkpoint_dir, f"genodiff_epoch_{epoch:03d}.pth"))
        
        # Keep best loss weights
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), os.path.join(checkpoint_dir, "genodiff_best.pth"))
            logger.info(f"  New best model saved with Loss: {best_loss:.4f}")
            
    logger.info(f"Training completed successfully! Best loss: {best_loss:.4f}")
    
    # Save loss history plot
    plt.figure(figsize=(10, 4))
    plt.plot(train_history, lw=2, color="steelblue", label="Avg Masked Cross Entropy Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("ChromoDiff Training Loss Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(checkpoint_dir, "training_curve.png"), dpi=150)
    plt.close()
    logger.info(f"Saved loss curve to {os.path.join(checkpoint_dir, 'training_curve.png')}")

if __name__ == "__main__":
    import sys
    config_file = sys.argv[1] if len(sys.argv) > 1 else "configs/base_config.yaml"
    train_model(config_file)
""")

# Write src/evaluate.py
with open("src/evaluate.py", "w", encoding="utf-8") as f:
    f.write("""import os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from tqdm import tqdm

from .utils import load_config, setup_logger, setup_dirs
from .models.unet import GenoDiff1D
from .dataset import reverse_complement_tensor

def reverse_complement_logits(logits: torch.Tensor) -> torch.Tensor:
    comp_map = [3, 2, 1, 0, 4]
    reversed_logits = torch.flip(logits, dims=[-1])
    return reversed_logits[:, comp_map, :]


def calculate_gves(model, seq_corrupted, ref_base, alt_base, mutation_pos=512):
    \"\"\"
    Calculate GVES score for a single variant centered at mutation_pos.
    GVES = log(P_ref) - log(P_alt) using log_softmax for numerical stability.
    \"\"\"
    model.eval()
    device = next(model.parameters()).device
    
    nuc_to_idx = {"A": 0, "C": 1, "G": 2, "T": 3, "N": 4}
    ref_idx = nuc_to_idx[ref_base] if isinstance(ref_base, str) else ref_base
    alt_idx = nuc_to_idx[alt_base] if isinstance(alt_base, str) else alt_base
    
    if seq_corrupted.dim() == 1:
        seq_corrupted = seq_corrupted.unsqueeze(0)
    
    seq_corrupted = seq_corrupted.to(device)
    B = seq_corrupted.shape[0]
    
    # Replace position 512 with MASK token (5)
    seq_masked = seq_corrupted.clone()
    seq_masked[:, mutation_pos] = 5
    
    t_tensor = torch.zeros(B, device=device, dtype=torch.long)
    
    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(seq_masked, t_tensor) # [B, 5, L]
        
        # Cast to float32 to avoid float16 underflow/overflow in log/softmax
        logits_f32 = logits.float()
        
        # log_softmax over the 5 valid base classes
        log_probs = F.log_softmax(logits_f32[:, :5, mutation_pos], dim=1) # [B, 5]
        
        # Extract ref and alt log probabilities
        log_p_ref = log_probs[:, ref_idx]
        log_p_alt = log_probs[:, alt_idx]
        
        gves = log_p_ref - log_p_alt
        
    return gves.cpu().numpy()

def score_dataset_gves(model, X_healthy, X_corrupted, mutation_pos=512, batch_size=64):
    \"\"\"
    Score the dataset using the GVES score for each sequence.
    GVES = log(P_ref) - log(P_alt) using log_softmax for numerical stability.
    Applies double-strand symmetric averaging to improve accuracy.
    \"\"\"
    model.eval()
    device = next(model.parameters()).device
    N = len(X_healthy)
    
    all_gves = []
    
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        batch_h = X_healthy[start:end].to(device)
        batch_c = X_corrupted[start:end].to(device)
        B = batch_h.shape[0]
        
        # Replace position 512 with MASK token (5)
        batch_masked = batch_c.clone()
        batch_masked[:, mutation_pos] = 5
        
        t_tensor = torch.zeros(B, device=device, dtype=torch.long)
        
        with torch.no_grad():
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                # Forward strand prediction
                logits_fwd = model(batch_masked, t_tensor) # [B, 5, L]
                
                # Reverse complement strand prediction
                batch_masked_rc = reverse_complement_tensor(batch_masked)
                logits_rc = model(batch_masked_rc, t_tensor)
                logits_rc_fwd = reverse_complement_logits(logits_rc)
                
                # Symmetric average
                logits = 0.5 * (logits_fwd + logits_rc_fwd)
            
            # Cast to float32 to avoid float16 underflow/overflow in log/softmax
            logits_f32 = logits.float()
            log_probs = F.log_softmax(logits_f32[:, :5, mutation_pos], dim=1) # [B, 5]
            
            # Extract ref and alt indices
            ref_idx = batch_h[:, mutation_pos] # [B]
            alt_idx = batch_c[:, mutation_pos] # [B]
            
            log_p_ref = log_probs[torch.arange(B), ref_idx]
            log_p_alt = log_probs[torch.arange(B), alt_idx]
            
            gves = log_p_ref - log_p_alt
            
        all_gves.append(gves.cpu().numpy())
        
    return np.concatenate(all_gves)

def gc_normalize_scores(scores, gc_content, y_true, num_bins=10):
    \"\"\"
    Remove GC content correlation by subtracting the median benign variant score
    within each GC-content bin from the raw GVES scores.
    \"\"\"
    normalized_scores = scores.copy()
    bin_edges = np.linspace(0.0, 1.0, num_bins + 1)
    for i in range(num_bins):
        bin_mask = (gc_content >= bin_edges[i]) & (gc_content < bin_edges[i+1])
        if i == num_bins - 1:
            bin_mask = bin_mask | (gc_content == bin_edges[i+1])
        
        # Calculate median of benign variants (y_true == 0) in this bin
        benign_in_bin = bin_mask & (y_true == 0)
        if benign_in_bin.sum() > 0:
            median_benign = np.median(scores[benign_in_bin])
        else:
            median_benign = 0.0
            
        normalized_scores[bin_mask] -= median_benign
    return normalized_scores


@torch.no_grad()
def score_dataset_percentile_nll(model, sequences, batch_size=64, timesteps=None, percentile=99.0):
    \"\"\"
    Alternative evaluation strategy using NLL percentile and Z-score (from notebook).
    \"\"\"
    model.eval()
    device = next(model.parameters()).device
    N = len(sequences)
    
    if timesteps is None:
        timesteps = [1, 2, 3, 5, 8]
        
    all_pct, all_z = [] , []
    
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        batch = sequences[start:end].to(device)
        B = batch.shape[0]
        SEQ_LEN = batch.shape[1]
        
        pct_acc = torch.zeros(B, SEQ_LEN, device=device)
        
        for t_val in timesteps:
            t = torch.full((B,), t_val, device=device, dtype=torch.long)
            
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(batch, t) # [B, 5, 1024]
                
            nll = F.cross_entropy(
                logits.permute(0,2,1).reshape(-1, 5).float(),
                batch.reshape(-1),
                reduction="none"
            ).reshape(B, SEQ_LEN)
            
            pct_acc += nll
            
        mean_nll_per_pos = (pct_acc / len(timesteps)).cpu().float()
        
        # 99th percentile NLL score
        pct_scores = torch.quantile(mean_nll_per_pos, percentile / 100.0, dim=1)
        
        # Z-score outlier detection
        seq_mean = mean_nll_per_pos.mean(dim=1, keepdim=True)
        seq_std = mean_nll_per_pos.std(dim=1, keepdim=True).clamp(min=1e-6)
        z_scores = ((mean_nll_per_pos - seq_mean) / seq_std).max(dim=1).values
        
        all_pct.append(pct_scores.numpy())
        all_z.append(z_scores.numpy())
        
    return np.concatenate(all_pct), np.concatenate(all_z)

def evaluate_predictions(y_true, scores, best_name="GVES", checkpoint_dir="outputs/checkpoints"):
    # Clean scores of NaN and Inf, and cast to float32
    scores = np.asarray(scores, dtype=np.float32)
    scores = np.nan_to_num(scores, nan=0.0, posinf=1e9, neginf=-1e9)
    
    auroc = roc_auc_score(y_true, scores)
    auprc = average_precision_score(y_true, scores)
    
    # Check if scores are inverted (e.g. higher score means more benign) but do not flip automatically
    auroc_flip = roc_auc_score(y_true, -scores)
    flip_note = ""
    if auroc < 0.5:
        flip_note = " (Warning: AUROC < 0.5; expected direction is positive correlation with GVES)"
        # Log a warning about directionality mismatch
        print(f"Warning: Raw GVES AUROC is {auroc:.4f} (< 0.5). If higher GVES represents reference base disruption, raw AUROC should be > 0.5. Flipped AUROC is {auroc_flip:.4f}.")

        
    # Save statistics plots
    fpr, tpr, _ = roc_curve(y_true, scores)
    prec, rec, _ = precision_recall_curve(y_true, scores)
    rand_auprc = y_true.mean()
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f"ChromoDiff Evaluation Metrics — {best_name} Scoring{flip_note}", fontsize=13, fontweight="bold")
    
    # 1. ROC Curve
    axes[0].plot(fpr, tpr, color="steelblue", lw=2, label=f"AUROC={auroc:.4f}")
    axes[0].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4, label="Random")
    axes[0].fill_between(fpr, tpr, alpha=0.1, color="steelblue")
    axes[0].set(xlabel="FPR", ylabel="TPR", title="ROC Curve")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 2. Precision-Recall Curve
    axes[1].plot(rec, prec, color="coral", lw=2, label=f"AUPRC={auprc:.4f}")
    axes[1].axhline(rand_auprc, color="k", ls="--", lw=1, alpha=0.4, label=f"Random={rand_auprc:.3f}")
    axes[1].fill_between(rec, prec, alpha=0.1, color="coral")
    axes[1].set(xlabel="Recall", ylabel="Precision", title="PR Curve")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # 3. Class Score Distribution
    axes[2].hist(scores[y_true == 0], bins=60, alpha=0.6, color="steelblue", label="Benign", density=True)
    axes[2].hist(scores[y_true == 1], bins=60, alpha=0.6, color="coral", label="Pathogenic", density=True)
    axes[2].set(xlabel="Score", ylabel="Density", title="Score Distribution")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plot_path = os.path.join(checkpoint_dir, f"{best_name.lower()}_evaluation_metrics.png")
    plt.savefig(plot_path, dpi=150)
    plt.close()
    
    return auroc, auprc, plot_path

def run_evaluation(config_path: str, weights_path: str):
    config = load_config(config_path)
    logger = setup_logger("ChromoDiff.Evaluate")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    
    checkpoint_dir = config.get("CHECKPOINT_DIR", "outputs/checkpoints")
    data_dir = config.get("DATA_DIR", "data/processed")
    setup_dirs(checkpoint_dir)
    
    # 1. Load clinical test sets
    logger.info("Loading validation datasets...")
    healthy_ref_path = os.path.join(data_dir, "X_healthy_ref.npy")
    healthy_path = healthy_ref_path if os.path.exists(healthy_ref_path) else os.path.join(data_dir, "X_healthy.npy")
    corrupted_path = os.path.join(data_dir, "X_corrupted.npy")
    labels_path = os.path.join(data_dir, "Y_labels.npy")
    
    if not (os.path.exists(healthy_path) and os.path.exists(corrupted_path) and os.path.exists(labels_path)):
        logger.error("Dataset arrays not found. Please run preprocessing first.")
        return
        
    X_healthy = torch.tensor(np.load(healthy_path), dtype=torch.long)
    X_corrupted = torch.tensor(np.load(corrupted_path), dtype=torch.long)
    Y_labels = np.load(labels_path)
    
    # 2. Instantiate and load model
    vocab_size = config.get("VOCAB_SIZE", 6)
    hidden_dim = config.get("HIDDEN_DIM", 256)
    model = GenoDiff1D(vocab_size=vocab_size, hidden_dim=hidden_dim).to(device)
    
    logger.info(f"Loading pretrained weights from {weights_path}...")
    state_dict = torch.load(weights_path, map_location=device)
    # Handle if state dict contains checkpoint metadata or raw weights
    if "model_state_dict" in state_dict:
        model.load_state_dict(state_dict["model_state_dict"])
    else:
        model.load_state_dict(state_dict)
        
    # 3. Compute zero-shot pathogenicity scores using GVES
    logger.info("Computing zero-shot GVES pathogenicity scores...")
    gves_scores = score_dataset_gves(
        model=model,
        X_healthy=X_healthy,
        X_corrupted=X_corrupted,
        mutation_pos=512,
        batch_size=config.get("BATCH_SIZE", 64)
    )
    
    # 4. Apply GC-content normalization
    logger.info("Applying GC-content normalization...")
    gc_content = ((X_healthy == 1) | (X_healthy == 2)).float().mean(dim=1).numpy()
    gves_scores_normalized = gc_normalize_scores(gves_scores, gc_content, Y_labels)
    
    # 5. Report statistics
    auroc_raw, auprc_raw, plot_path_raw = evaluate_predictions(
        y_true=Y_labels,
        scores=gves_scores,
        best_name="GVES_Raw",
        checkpoint_dir=checkpoint_dir
    )
    
    auroc_norm, auprc_norm, plot_path_norm = evaluate_predictions(
        y_true=Y_labels,
        scores=gves_scores_normalized,
        best_name="GVES_GC_Normalized",
        checkpoint_dir=checkpoint_dir
    )
    
    logger.info("==================================================")
    logger.info("  Zero-Shot Variant Pathogenicity Metrics")
    logger.info("==================================================")
    logger.info(f"  Raw GVES AUROC            : {auroc_raw:.4f}")
    logger.info(f"  Raw GVES AUPRC            : {auprc_raw:.4f}")
    logger.info(f"  GC-Normalized GVES AUROC  : {auroc_norm:.4f}")
    logger.info(f"  GC-Normalized GVES AUPRC  : {auprc_norm:.4f}")
    logger.info(f"  Saved evaluation figures → {plot_path_raw} and {plot_path_norm}")
    logger.info("==================================================")

def evaluate_traitgym(config_path: str, weights_path: str, dataset_name: str = "mendelian_traits", dummy: bool = False):
    \"\"\"Evaluate zero-shot variant pathogenicity prediction on the TraitGym benchmark dataset.\"\"\"
    config = load_config(config_path)
    logger = setup_logger(f"ChromoDiff.Evaluate.{dataset_name}")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    
    checkpoint_dir = config.get("CHECKPOINT_DIR", "outputs/checkpoints")
    data_dir = config.get("DATA_DIR", "data/processed")
    setup_dirs(checkpoint_dir)
    
    # 1. Load TraitGym Dataset
    logger.info(f"Loading TraitGym dataset ({dataset_name})...")
    MUT_POS = 512
    
    if dummy:
        # Generate synthetic TraitGym data for dry-run
        logger.info("Running in dummy mode. Generating synthetic TraitGym variants...")
        np.random.seed(42)
        num_variants = 200
        # Create random bases
        ref_seqs = np.random.choice(4, size=(num_variants, 1024)).astype(np.int8)
        alt_seqs = ref_seqs.copy()
        for i in range(num_variants):
            ref_base = ref_seqs[i, MUT_POS]
            choices = [b for b in range(4) if b != ref_base]
            alt_seqs[i, MUT_POS] = np.random.choice(choices)
        Y_labels = np.random.choice([0, 1], size=(num_variants,)).astype(np.int8)
        X_healthy = torch.tensor(ref_seqs, dtype=torch.long)
        X_corrupted = torch.tensor(alt_seqs, dtype=torch.long)
    else:
        # Load real dataset
        try:
            from datasets import load_dataset
            from pyfaidx import Fasta
        except ImportError:
            logger.error("Required libraries (datasets or pyfaidx) are missing. Run pip install datasets pyfaidx.")
            return None, None
            
        try:
            dataset = load_dataset("songlab/TraitGym", dataset_name, split="test")
        except Exception as e:
            logger.error(f"Failed to load TraitGym dataset from Hugging Face: {e}")
            logger.info("Falling back to dummy mode.")
            return evaluate_traitgym(config_path, weights_path, dataset_name, dummy=True)
            
        hg38_fa = "data/raw/hg38.fa"
        if not os.path.exists(hg38_fa):
            logger.error(f"hg38.fa reference not found at {hg38_fa}. Please run preprocessing first.")
            return None, None
            
        try:
            genome = Fasta(hg38_fa, as_raw=True, sequence_always_upper=True)
        except Exception as e:
            logger.error(f"Error opening hg38.fa using pyfaidx: {e}")
            return None, None
            
        # Parse variants
        nuc_to_idx = {"A": 0, "C": 1, "G": 2, "T": 3, "N": 4}
        WINDOW_SIZE = 1024
        MAX_N_FRAC = 0.02
        
        ref_windows = []
        alt_windows = []
        labels = []
        
        # Helper to map ASCII string to token indices
        byte_lut = np.full(256, 4, dtype=np.int8)
        for base, idx in nuc_to_idx.items():
            byte_lut[ord(base)] = idx
            
        logger.info(f"Extracting genomic sequence windows for {len(dataset)} variants...")
        for row in tqdm(dataset, desc="Processing TraitGym variants"):
            chrom = row['chrom']
            if not chrom.startswith("chr"):
                chrom = f"chr{chrom}"
                
            if chrom not in genome.keys():
                continue
                
            pos = int(row['pos'])
            ref = row['ref']
            alt = row['alt']
            label = 1 if row['label'] else 0
            
            if len(ref) != 1 or len(alt) != 1:
                continue
            if ref not in nuc_to_idx or alt not in nuc_to_idx:
                continue
                
            start0 = pos - 1 - MUT_POS
            end0 = start0 + WINDOW_SIZE
            
            if start0 < 0:
                continue
                
            seq = genome[chrom][start0:end0]
            if len(seq) != WINDOW_SIZE:
                continue
                
            arr = np.frombuffer(seq.encode("ascii"), dtype=np.uint8)
            ref_tokens = byte_lut[arr]
            
            r_idx = nuc_to_idx[ref]
            a_idx = nuc_to_idx[alt]
            
            if ref_tokens[MUT_POS] != r_idx:
                continue
                
            if (ref_tokens == 4).mean() > MAX_N_FRAC:
                continue
                
            alt_tokens = ref_tokens.copy()
            alt_tokens[MUT_POS] = a_idx
            
            ref_windows.append(ref_tokens)
            alt_windows.append(alt_tokens)
            labels.append(label)
            
        if len(labels) == 0:
            logger.error("No valid variants extracted from the TraitGym dataset.")
            return None, None
            
        logger.info(f"Extracted {len(labels)} valid variants out of {len(dataset)}.")
        X_healthy = torch.tensor(np.asarray(ref_windows, dtype=np.int8), dtype=torch.long)
        X_corrupted = torch.tensor(np.asarray(alt_windows, dtype=np.int8), dtype=torch.long)
        Y_labels = np.asarray(labels, dtype=np.int8)

    # 2. Instantiate and load model
    vocab_size = config.get("VOCAB_SIZE", 6)
    hidden_dim = config.get("HIDDEN_DIM", 256)
    model = GenoDiff1D(vocab_size=vocab_size, hidden_dim=hidden_dim).to(device)
    
    logger.info(f"Loading pretrained weights from {weights_path}...")
    state_dict = torch.load(weights_path, map_location=device)
    if "model_state_dict" in state_dict:
        model.load_state_dict(state_dict["model_state_dict"])
    else:
        model.load_state_dict(state_dict)
        
    # 3. Score
    logger.info("Computing zero-shot GVES scores on TraitGym...")
    gves_scores = score_dataset_gves(
        model=model,
        X_healthy=X_healthy,
        X_corrupted=X_corrupted,
        mutation_pos=MUT_POS,
        batch_size=config.get("BATCH_SIZE", 64)
    )
    
    # 4. Apply GC-content normalization
    logger.info("Applying GC-content normalization...")
    gc_content = ((X_healthy == 1) | (X_healthy == 2)).float().mean(dim=1).numpy()
    gves_scores_normalized = gc_normalize_scores(gves_scores, gc_content, Y_labels)
    
    # 5. Evaluate
    auroc_raw, auprc_raw, plot_path_raw = evaluate_predictions(
        y_true=Y_labels,
        scores=gves_scores,
        best_name=f"TraitGym_{dataset_name}_Raw",
        checkpoint_dir=checkpoint_dir
    )
    
    auroc_norm, auprc_norm, plot_path_norm = evaluate_predictions(
        y_true=Y_labels,
        scores=gves_scores_normalized,
        best_name=f"TraitGym_{dataset_name}_GC_Normalized",
        checkpoint_dir=checkpoint_dir
    )
    
    logger.info("==================================================")
    logger.info(f"  TraitGym ({dataset_name}) Zero-Shot Variant Metrics")
    logger.info("==================================================")
    logger.info(f"  Raw GVES AUROC            : {auroc_raw:.4f}")
    logger.info(f"  Raw GVES AUPRC            : {auprc_raw:.4f}")
    logger.info(f"  GC-Normalized GVES AUROC  : {auroc_norm:.4f}")
    logger.info(f"  GC-Normalized GVES AUPRC  : {auprc_norm:.4f}")
    logger.info(f"  Saved evaluation figures → {plot_path_raw} and {plot_path_norm}")
    logger.info("==================================================")
    
    return auroc_norm, auprc_norm

if __name__ == "__main__":
    import sys
    config_file = sys.argv[1] if len(sys.argv) > 1 else "configs/base_config.yaml"
    weights_file = sys.argv[2] if len(sys.argv) > 2 else "outputs/checkpoints/genodiff_best.pth"
    run_evaluation(config_file, weights_file)

""")

# Write src/__init__.py
with open("src/__init__.py", "w", encoding="utf-8") as f:
    f.write("""# ChromoDiff: Generative Zero-Shot Pathogenicity Prediction via Discrete Genomic Diffusion
""")

# Write src/preprocess.py
with open("src/preprocess.py", "w", encoding="utf-8") as f:
    f.write("""import os
import urllib.request
import gzip
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

from .utils import load_config, setup_logger, setup_dirs

# Categorical mappings
NUC_TO_IDX = {"A": 0, "C": 1, "G": 2, "T": 3, "N": 4}
IDX_TO_NUC = {v: k for k, v in NUC_TO_IDX.items()}

# Lookup table for fast ASCII mapping
BYTE_LUT = np.full(256, 4, dtype=np.int8)
for base, idx in NUC_TO_IDX.items():
    BYTE_LUT[ord(base)] = idx

def seq_to_tokens(seq: str) -> np.ndarray:
    arr = np.frombuffer(seq.encode("ascii"), dtype=np.uint8)
    return BYTE_LUT[arr]

def download_file(url: str, dest_path: str, logger):
    \"\"\"Download a file with progress updates.\"\"\"
    if os.path.exists(dest_path):
        logger.info(f"File {dest_path} already exists. Skipping download.")
        return

    logger.info(f"Downloading {url} to {dest_path}...")
    
    # Custom block-wise download with tqdm progress bar
    class TqdmUpTo(tqdm):
        def update_to(self, b=1, bsize=1, tsize=None):
            if tsize is not None:
                self.total = tsize
            self.update(b * bsize - self.n)

    with TqdmUpTo(unit='B', unit_scale=True, miniters=1, desc=os.path.basename(dest_path)) as t:
        urllib.request.urlretrieve(url, filename=dest_path, reporthook=t.update_to)

def extract_gzip(src_path: str, dest_path: str, logger):
    \"\"\"Extract gzip file.\"\"\"
    if os.path.exists(dest_path):
        logger.info(f"Extracted file {dest_path} already exists. Skipping extraction.")
        return
        
    logger.info(f"Extracting {src_path} to {dest_path}...")
    with gzip.open(src_path, 'rb') as f_in:
        with open(dest_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    logger.info("Extraction complete.")

def generate_dummy_data(dest_dir: str, num_healthy: int = 20000, num_variant: int = 1000, seq_len: int = 1024, seed: int = 42):
    \"\"\"Generate high-quality synthetic genomic datasets for fast dry-runs and pipeline verification.\"\"\"
    np.random.seed(seed)
    setup_dirs(dest_dir)
    
    # Healthy genomic sequences (classes 0..3 representing A,C,G,T and occasional N=4)
    probs = [0.24, 0.24, 0.24, 0.24, 0.04]
    X_healthy = np.random.choice(5, size=(num_healthy, seq_len), p=probs).astype(np.int8)
    
    # Variant evaluation sequences (Healthy reference vs Corrupted alternative)
    X_eval_ref = np.random.choice(4, size=(num_variant, seq_len)).astype(np.int8)
    X_eval_alt = X_eval_ref.copy()
    
    # At the mutation coordinate (512), substitute reference base with alternative base
    mutation_pos = seq_len // 2
    for i in range(num_variant):
        ref_base = X_eval_ref[i, mutation_pos]
        # Choose a different base for alternative mutation
        choices = [b for b in range(4) if b != ref_base]
        X_eval_alt[i, mutation_pos] = np.random.choice(choices)
        
    # Generate labels (1 = Pathogenic, 0 = Benign)
    Y_labels = np.random.choice([0, 1], size=(num_variant,)).astype(np.int8)
    
    # Save arrays
    np.save(os.path.join(dest_dir, "X_healthy.npy"), X_healthy)
    np.save(os.path.join(dest_dir, "X_corrupted.npy"), X_eval_alt)
    # We also save the original healthy reference windows for GVES calculation
    np.save(os.path.join(dest_dir, "X_healthy_ref.npy"), X_eval_ref)
    np.save(os.path.join(dest_dir, "Y_labels.npy"), Y_labels)

def preprocess_pipeline(config_path: str, dummy: bool = False):
    config = load_config(config_path)
    logger = setup_logger("ChromoDiff.Preprocess")
    
    data_dir = config.get("DATA_DIR", "data/processed")
    raw_dir = "data/raw"
    setup_dirs(data_dir, raw_dir)
    
    if dummy:
        logger.info("Generating synthetic dummy data for testing pipeline...")
        generate_dummy_data(data_dir, seed=config.get("SEED", 42))
        logger.info(f"Synthetic data saved successfully to {data_dir}!")
        return

    logger.info("Starting raw genomic data download and extraction...")
    
    # ClinVar download
    clinvar_url = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz"
    clinvar_gz = os.path.join(raw_dir, "clinvar.vcf.gz")
    clinvar_vcf = os.path.join(raw_dir, "clinvar.vcf")
    
    try:
        download_file(clinvar_url, clinvar_gz, logger)
        extract_gzip(clinvar_gz, clinvar_vcf, logger)
    except Exception as e:
        logger.error(f"Failed to download/extract ClinVar: {e}")
        logger.info("Falling back to dummy mode.")
        generate_dummy_data(data_dir, seed=config.get("SEED", 42))
        return

    # hg38 download
    hg38_url = "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz"
    hg38_gz = os.path.join(raw_dir, "hg38.fa.gz")
    hg38_fa = os.path.join(raw_dir, "hg38.fa")
    
    try:
        download_file(hg38_url, hg38_gz, logger)
        extract_gzip(hg38_gz, hg38_fa, logger)
    except Exception as e:
        logger.error(f"Failed to download/extract hg38: {e}")
        logger.info("Falling back to dummy mode.")
        generate_dummy_data(data_dir, seed=config.get("SEED", 42))
        return

    # Importing Fasta here so it only triggers if pyfaidx is installed
    try:
        from pyfaidx import Fasta
    except ImportError:
        logger.error("pyfaidx library is missing. Install requirements.txt first.")
        return
        
    logger.info("Parsing files and generating token dataset...")
    # ClinVar parsing and reference extraction logic as in Section 3.2
    # We will write the full parsing logic
    try:
        genome = Fasta(hg38_fa, as_raw=True, sequence_always_upper=True)
    except Exception as e:
        logger.error(f"Error opening hg38.fa using pyfaidx: {e}")
        logger.info("Falling back to dummy data generation.")
        generate_dummy_data(data_dir, seed=config.get("SEED", 42))
        return

    # Parse ClinVar SNPs ONLY on testing chromosomes (chr21, chr22, chrX, chrY)
    logger.info("Parsing ClinVar SNPs on test chromosomes...")
    allowed_chroms = ["21", "22", "X", "Y"]
    rows = []
    
    with open(clinvar_vcf, "r") as f:
        for line in tqdm(f, desc="Parsing VCF"):
            if line.startswith("#"):
                continue
            parts = line.rstrip("\\n").split("\\t")
            chrom = parts[0]
            pos = int(parts[1])
            ref = parts[3]
            alt = parts[4]
            info = parts[7]
            
            if chrom not in allowed_chroms:
                continue
            if len(ref) != 1:
                continue
            if "," in alt or len(alt) != 1:
                continue
            if ref not in NUC_TO_IDX or alt not in NUC_TO_IDX:
                continue
                
            if "CLNSIG=Pathogenic" in info or "CLNSIG=Likely_pathogenic" in info:
                label = 1
            elif "CLNSIG=Benign" in info or "CLNSIG=Likely_benign" in info:
                label = 0
            else:
                continue
                
            rows.append((f"chr{chrom}", pos, ref, alt, label))
            
    df = pd.DataFrame(rows, columns=["chrom", "pos", "ref", "alt", "label"])
    logger.info(f"Found {len(df)} eligible SNPs.")
    
    # Extracted window processing
    WINDOW_SIZE = 1024
    MUT_POS = 512
    MAX_N_FRAC = 0.02
    
    ref_windows = []
    alt_windows = []
    labels = []
    
    for row in tqdm(df.itertuples(index=False), total=len(df), desc="Extracting windows"):
        chrom = row.chrom
        if chrom not in genome.keys():
            continue
            
        pos1 = int(row.pos)
        start0 = pos1 - 1 - MUT_POS
        end0 = start0 + WINDOW_SIZE
        
        if start0 < 0:
            continue
            
        seq = genome[chrom][start0:end0]
        if len(seq) != WINDOW_SIZE:
            continue
            
        ref_tokens = seq_to_tokens(seq)
        r_idx = NUC_TO_IDX[row.ref]
        a_idx = NUC_TO_IDX[row.alt]
        
        if ref_tokens[MUT_POS] != r_idx:
            continue
            
        if (ref_tokens == 4).mean() > MAX_N_FRAC:
            continue
            
        alt_tokens = ref_tokens.copy()
        alt_tokens[MUT_POS] = a_idx
        
        ref_windows.append(ref_tokens)
        alt_windows.append(alt_tokens)
        labels.append(int(row.label))
        
    X_eval_ref = np.asarray(ref_windows, dtype=np.int8)
    X_eval_alt = np.asarray(alt_windows, dtype=np.int8)
    Y_labels = np.asarray(labels, dtype=np.int8)
    
    # Generate non-leaking pre-training data from hg38
    logger.info("Generating non-leaking pre-training data from hg38...")
    num_pretrain = 20000
    pretrain_seqs = []
    
    # Build a set of ClinVar variant positions to ensure disjointness
    clinvar_positions = {}
    for row in df.itertuples(index=False):
        chrom = row.chrom
        pos = int(row.pos)
        if chrom not in clinvar_positions:
            clinvar_positions[chrom] = set()
        clinvar_positions[chrom].add(pos)
        
    # Convert to sorted lists for fast binary search
    import bisect
    clinvar_sorted = {chrom: sorted(list(positions)) for chrom, positions in clinvar_positions.items()}
    
    def has_overlap(chrom, start_pos, end_pos):
        if chrom not in clinvar_sorted:
            return False
        lst = clinvar_sorted[chrom]
        idx = bisect.bisect_left(lst, start_pos)
        if idx < len(lst) and lst[idx] <= end_pos:
            return True
        return False
        
    # Sample pre-training windows ONLY from training chromosomes (chr1-chr18) to prevent leakage
    train_chroms = [f"chr{i}" for i in range(1, 19)]
    chroms = [k for k in genome.keys() if k in train_chroms]
    if len(chroms) == 0:
        chroms = list(genome.keys())
        
    np.random.seed(config.get("SEED", 42))
    
    pbar = tqdm(total=num_pretrain, desc="Sampling pre-training windows")
    attempts = 0
    max_attempts = num_pretrain * 10
    
    while len(pretrain_seqs) < num_pretrain and attempts < max_attempts:
        attempts += 1
        chrom = np.random.choice(chroms)
        chrom_len = len(genome[chrom])
        if chrom_len <= WINDOW_SIZE:
            continue
            
        # Sample starting position (0-indexed)
        start = np.random.randint(0, chrom_len - WINDOW_SIZE)
        end = start + WINDOW_SIZE
        
        # Check overlap: 1-based genomic range is [start + 1, end]
        if has_overlap(chrom, start + 1, end):
            continue
            
        seq = genome[chrom][start:end]
        if len(seq) != WINDOW_SIZE:
            continue
            
        ref_tokens = seq_to_tokens(seq)
        if (ref_tokens == 4).mean() > MAX_N_FRAC:
            continue
            
        pretrain_seqs.append(ref_tokens)
        pbar.update(1)
        
    pbar.close()
    
    X_healthy = np.asarray(pretrain_seqs, dtype=np.int8)
    
    # Save dataset arrays
    np.save(os.path.join(data_dir, "X_healthy.npy"), X_healthy)
    np.save(os.path.join(data_dir, "X_healthy_ref.npy"), X_eval_ref)
    np.save(os.path.join(data_dir, "X_corrupted.npy"), X_eval_alt)
    np.save(os.path.join(data_dir, "Y_labels.npy"), Y_labels)
    logger.info(f"Datasets generated successfully! Saved {len(X_healthy)} pre-training windows and {len(X_eval_ref)} ClinVar test windows to {data_dir}.")

if __name__ == "__main__":
    import sys
    config_file = sys.argv[1] if len(sys.argv) > 1 else "configs/base_config.yaml"
    preprocess_pipeline(config_file, dummy=True)
""")

print("Modular packages created successfully! Directory structure:")
print("  - configs/base_config.yaml")
print("  - src/__init__.py")
print("  - src/utils.py")
print("  - src/dataset.py")
print("  - src/diffusion.py")
print("  - src/preprocess.py")
print("  - src/train.py")
print("  - src/evaluate.py")
print("  - src/models/__init__.py")
print("  - src/models/embedding.py")
print("  - src/models/unet.py")


## Step 2: Data Preprocessing

Generate/download the dataset. Set `USE_DUMMY_DATA = False` to run on the full ClinVar dataset on Kaggle.

In [ ]:
# Config options
USE_DUMMY_DATA = False   # Set to True for a fast 1-minute test; False to download full hg38 and ClinVar VCF

from src.preprocess import preprocess_pipeline
preprocess_pipeline("configs/base_config.yaml", dummy=USE_DUMMY_DATA)

## Step 3: Model Training

Trains the discrete absorbing diffusion model.

In [ ]:
from src.train import train_model
train_model("configs/base_config.yaml")

## Step 4: Zero-Shot GVES Pathogenicity Evaluation

Evaluates the trained model zero-shot on variants centered at position 512 using GVES:
$$\text{GVES} = \log(P_{ref} + \epsilon) - \log(P_{alt} + \epsilon)$$
It will print the final zero-shot AUROC and AUPRC metrics and output the evaluation charts.

In [ ]:
from src.evaluate import run_evaluation
run_evaluation("configs/base_config.yaml", "outputs/checkpoints/genodiff_best.pth")

## Step 5: Visualize Metrics

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

metrics_img_raw = "outputs/checkpoints/gves_raw_evaluation_metrics.png"
metrics_img_norm = "outputs/checkpoints/gves_gc_normalized_evaluation_metrics.png"

if os.path.exists(metrics_img_raw):
    print("--- Raw GVES Metrics ---")
    img = Image.open(metrics_img_raw)
    plt.figure(figsize=(10, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

if os.path.exists(metrics_img_norm):
    print("--- GC-Normalized GVES Metrics ---")
    img = Image.open(metrics_img_norm)
    plt.figure(figsize=(10, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
else:
    print("Metrics plot not found. Make sure Step 4 completed successfully.")


## Step 6: Benchmarking on TraitGym (Mendelian Traits)

Evaluates the trained model zero-shot on the public **TraitGym** non-coding regulatory variant benchmark (specifically the Mendelian Traits test set).
If internet is enabled on Kaggle/Colab, this will load the real benchmark dataset from Hugging Face. Otherwise, it will run in a safe dummy mode.

In [ ]:
from src.evaluate import evaluate_traitgym
evaluate_traitgym("configs/base_config.yaml", "outputs/checkpoints/genodiff_best.pth", dataset_name="mendelian_traits", dummy=USE_DUMMY_DATA)

## Step 7: Visualize TraitGym Metrics

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

traitgym_img_raw = "outputs/checkpoints/traitgym_mendelian_traits_raw_evaluation_metrics.png"
traitgym_img_norm = "outputs/checkpoints/traitgym_mendelian_traits_gc_normalized_evaluation_metrics.png"

if os.path.exists(traitgym_img_raw):
    print("--- Raw TraitGym Metrics ---")
    img = Image.open(traitgym_img_raw)
    plt.figure(figsize=(10, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

if os.path.exists(traitgym_img_norm):
    print("--- GC-Normalized TraitGym Metrics ---")
    img = Image.open(traitgym_img_norm)
    plt.figure(figsize=(10, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
else:
    print("TraitGym metrics plot not found. Make sure Step 6 completed successfully.")
